# Redes Neuronales II — Clasificación Multiclase con TensorFlow + Keras
## Dataset MNIST (reconocimiento de dígitos escritos a mano)

**Importante:** esta red utiliza **únicamente capas densas (`Dense`)**, sin capas convolucionales, tal como lo exige la actividad.

Arquitectura: `784 (entrada aplanada 28x28) -> 128 (ReLU) -> 64 (ReLU) -> 10 (Softmax)`

- Las capas ocultas usan **ReLU** como función de activación.
- La capa de salida usa **Softmax**, adecuada para clasificación **multiclase** (10 dígitos: 0-9), ya que convierte los logits en una distribución de probabilidad sobre las clases.
- El entrenamiento (backpropagation + descenso de gradiente) es gestionado automáticamente por Keras mediante el optimizador **Adam** y la función de pérdida **sparse_categorical_crossentropy**.

> Ejecutar este notebook en Google Colab: la primera celda descargará el dataset MNIST automáticamente (requiere conexión a internet, disponible en Colab).

In [ ]:
!pip install -q tensorflow

In [ ]:
"""
04 - Clasificación multiclase con TensorFlow + Keras sobre el dataset MNIST
IMPORTANTE: Solo capas densas (Dense) - NO se usan redes convolucionales.
Ejecutar este script en Google Colab (requiere descargar MNIST de internet).
"""
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("Versión de TensorFlow:", tf.__version__)

# 1. Cargar y preparar los datos
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Normalizar los píxeles al rango [0, 1] y aplanar las imágenes 28x28 -> 784
X_train = X_train.reshape(-1, 28 * 28).astype("float32") / 255.0
X_test = X_test.reshape(-1, 28 * 28).astype("float32") / 255.0

print("Forma de X_train:", X_train.shape)
print("Forma de X_test:", X_test.shape)

# 2. Definir la arquitectura del modelo (perceptrón multicapa)
modelo = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation="relu", name="capa_oculta_1"),
    layers.Dense(64, activation="relu", name="capa_oculta_2"),
    layers.Dense(10, activation="softmax", name="capa_salida"),
])

modelo.summary()

# 3. Compilar el modelo
modelo.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# 4. Entrenar el modelo (backpropagation gestionado internamente por Keras)
historial = modelo.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=2,
)

# 5. Evaluar el modelo sobre el conjunto de prueba
perdida_test, exactitud_test = modelo.evaluate(X_test, y_test, verbose=0)
print(f"\nPérdida en prueba: {perdida_test:.4f}")
print(f"Exactitud en prueba: {exactitud_test:.4f}")

# 6. Matriz de confusión y reporte
from sklearn.metrics import confusion_matrix, classification_report

y_pred = np.argmax(modelo.predict(X_test), axis=1)
print("\nMatriz de confusión:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred))

# 7. Gráficas de entrenamiento (evidencia de ejecución)
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(historial.history["accuracy"], label="entrenamiento")
axs[0].plot(historial.history["val_accuracy"], label="validación")
axs[0].set_title("Exactitud por época - Keras/MNIST")
axs[0].set_xlabel("Época")
axs[0].set_ylabel("Exactitud")
axs[0].legend()
axs[0].grid(True)

axs[1].plot(historial.history["loss"], label="entrenamiento")
axs[1].plot(historial.history["val_loss"], label="validación")
axs[1].set_title("Pérdida por época - Keras/MNIST")
axs[1].set_xlabel("Época")
axs[1].set_ylabel("Pérdida")
axs[1].legend()
axs[1].grid(True)

plt.tight_layout()
plt.savefig("evidencia_04_tensorflow_keras_mnist.png", dpi=150)
print("\nGráfica guardada como evidencia_04_tensorflow_keras_mnist.png")

# 8. Guardar el modelo entrenado (opcional, útil para el repositorio de GitHub)
modelo.save("modelo_keras_mnist.h5")
print("Modelo guardado como modelo_keras_mnist.h5")


### Resultados esperados y análisis

Con esta arquitectura (dos capas ocultas densas, ReLU + Softmax, optimizador Adam) es normal obtener una **exactitud de prueba superior al 97%** tras 10 épocas de entrenamiento, sin usar capas convolucionales. La curva de exactitud debe mostrar una tendencia creciente y estabilizarse, mientras que la curva de pérdida decrece de forma consistente tanto en entrenamiento como en validación (si ambas curvas se separan mucho, sería indicio de sobreajuste).

**Pega aquí las capturas de pantalla de la ejecución en tu Colab** (resumen del modelo, progreso del entrenamiento por época, matriz de confusión y las gráficas generadas) como evidencia de ejecución para el documento técnico.
